# Trend deep dive

Deliverables 3-7: colour identity drift, pie-break tracking, rarity migration, type-line
crossover and the complexity proxy. Requires `mtg-analysis build` to have run.

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import polars as pl

from mtg_analysis.analysis.color_drift import color_drift, keyword_vectors, plot_color_drift
from mtg_analysis.analysis.complexity import complexity_by_color, plot_complexity
from mtg_analysis.analysis.db import get_connection
from mtg_analysis.analysis.pie_break import pie_break, plot_pie_break
from mtg_analysis.analysis.rarity_migration import plot_rarity_migration, rarity_migration
from mtg_analysis.analysis.type_crossover import plot_type_crossover, type_crossover
from mtg_analysis.config import PeriodGroupConfig, load_config

config = load_config("../config/config.yaml")
# Grouping consecutive sets smooths the small-set noise these views are sensitive to.
con = get_connection("../" / config.paths.processed_dir,
                     PeriodGroupConfig(mode="rolling_sets", group_size=5))

## Colour identity drift (deliverable 3)

Each colour's keyword mix is normalized per period, then compared by cosine similarity to
the previous period and to the colour's all-time baseline. Lower means the toolbox moved.

In [ ]:
drift = color_drift(con)
fig, axes = plt.subplots(2, 1, figsize=(11, 9))
plot_color_drift(drift, metric="cosine_vs_baseline", ax=axes[0])
plot_color_drift(drift, metric="cosine_vs_previous", ax=axes[1])
fig.tight_layout()
plt.show()
drift.group_by("color").agg(pl.col("cosine_vs_baseline").mean()).sort("cosine_vs_baseline")

In [ ]:
# The raw vectors, if you want to see which keywords drove a shift.
keyword_vectors(con).filter(pl.col("color") == "R").sort("freq", descending=True).head(10)

## Pie-break tracking (deliverable 4)

Finds the colour that historically owned a keyword, then tracks how much of it other
colours have taken.

In [ ]:
breaks = pie_break(con, "Double strike")
plot_pie_break(breaks)
plt.show()
breaks.select("period", "color", "share", "challenger_share").tail(10)

## Rarity migration (deliverable 5)

In [ ]:
rarity = rarity_migration(con, "Haste")
plot_rarity_migration(rarity)
plt.show()
rarity.tail()

## Type-line crossover (deliverable 6)

In [ ]:
crossover = type_crossover(con, "Haste")
plot_type_crossover(crossover)
plt.show()

## Complexity proxy (deliverable 7)

Mean oracle text length and mean keyword count per card, weighted by colour.

In [ ]:
complexity = complexity_by_color(con)
fig, axes = plt.subplots(2, 1, figsize=(11, 9))
plot_complexity(complexity, metric="mean_text_length", ax=axes[0])
plot_complexity(complexity, metric="mean_keywords", ax=axes[1])
fig.tight_layout()
plt.show()